# Companion Notebook 10: Perplexity and Model Confusion Verification

This notebook verifies the mathematical relationship between **Cross-Entropy Loss** and **Perplexity (PPL)**, executing a step-by-step verification on sample probabilities.


## 1. Hand-Calculation Verification
We compute the perplexity of a generated sequence of length $L=2$, where correct target token probabilities are $P(x_1) = 0.50$ and $P(x_2 | x_1) = 0.20$.


In [1]:
import numpy as np
import torch

# Probabilities of the correct tokens at each step
probs = np.array([0.50, 0.20])

# Step 1: Calculate negative log probabilities
log_probs = np.log(probs)
neg_log_probs = -log_probs
print("Step 1: Negative Log Probabilities (Cross-Entropy Loss per token):")
print(neg_log_probs)

# Step 2: Calculate average loss
avg_loss = np.mean(neg_log_probs)
print(f"\nStep 2: Average Cross-Entropy Loss: {avg_loss:.4f}")

# Step 3: Compute Perplexity
perplexity = np.exp(avg_loss)
print(f"Step 3: Perplexity: {perplexity:.4f}")

# Expected value verification
expected_ppl = 3.1623
print("\nMatches study guide predictions (3.1623)?")
print(np.isclose(perplexity, expected_ppl, atol=1e-3))


Step 1: Negative Log Probabilities (Cross-Entropy Loss per token):
[0.69314718 1.60943791]

Step 2: Average Cross-Entropy Loss: 1.1513
Step 3: Perplexity: 3.1623

Matches study guide predictions (3.1623)?
True


### Explanation of Outputs (Hand Calculation Verification)
- **Perplexity**: The resulting perplexity $\approx 3.1623$, indicating that at each step, the model is on average as confused as choosing uniformly among 3.16 options.
- **Verification check**: Returns `True` showing that the manual calculations and programmatic evaluations are identical.


## 2. PyTorch Cross-Entropy Loss to Perplexity Verification
We create dummy logits and targets in PyTorch, calculate average cross-entropy loss, and verify its exponentiated relation to perplexity.


In [2]:
import torch.nn as nn

# Vocabulary size |V| = 4, Sequence length L = 3, Batch size b = 1
vocab_size = 4
L_seq = 3

# Raw logits output by a mock model: shape [batch, seq_len, vocab_size]
# We transpose to [batch, vocab_size, seq_len] for PyTorch CrossEntropyLoss
logits = torch.tensor([[[2.0, 1.0, 0.1, -0.5],
                        [0.5, 3.0, 1.0, -1.0],
                        [1.0, 1.5, 4.0, 0.2]]], dtype=torch.float32)

# Correct target token IDs
targets = torch.tensor([[0, 1, 2]], dtype=torch.long)

# Loss module
loss_fn = nn.CrossEntropyLoss(reduction='mean')

# Reshape logits: [batch, vocab_size, seq_len]
logits_reshaped = logits.transpose(1, 2)

# Compute average cross entropy loss
loss_val = loss_fn(logits_reshaped, targets)

# Compute perplexity
ppl_val = torch.exp(loss_val)

print("Average PyTorch Cross Entropy Loss:", loss_val.item())
print("Calculated Perplexity Value:       ", ppl_val.item())


Average PyTorch Cross Entropy Loss: 0.27494099736213684
Calculated Perplexity Value:        1.316452980041504


### Explanation of Outputs (PyTorch CrossEntropy to Perplexity)
- **Average Loss**: Computes the mean cross-entropy loss over the sequence indices.
- **Calculated Perplexity**: Evaluates to the exponentiated loss, matching the direct mapping relation $\text{PPL} = e^{H(X)}$.
